# QueenAgent — Colab

Bu defterin işi kurmak ve sunmak: Drive'ı bağlar → repoyu klonlar → **Flask** arayüzünü servis
eder → bir link basar. Uygulamanın kendisi linkin arkasında çalışıyor.

Projelerin, sohbetlerin ve üretilen dosyaların hepsi **kendi Drive'ında** duruyor — hangi klasörde
olduğunu aşağıdaki CONFIG hücresi söyler (`DRIVE_FOLDER`) ve çalışınca yazdırır. Runtime kapansa da
kaybolmuyorlar.

In [ ]:
# === CONFIG ===
# Drive is mounted on the very first line: the permission window has to appear in the first
# second (NOTEBOOK-STANDARD). One that shows up forty seconds in waits on a user who has already
# walked away.
import os

from google.colab import drive, userdata

drive.mount("/content/drive")

# Everything that gets changed by hand lives here and nowhere else.
DRIVE_FOLDER = "queenAgent"          # proje kökü (MyDrive altında) — adı buradan değiştir
REPO         = "AltanBaysal/Internal-tools"
# Released work will land on main; until this notebook is merged there, main does not carry it,
# and a branch that does not exist fails at clone time with nothing that explains why.
BRANCH       = "feat/queenagent-colab"
CLONE_DIR    = "/content/Internal-tools"
APP_DIR      = f"{CLONE_DIR}/queen-agent"
APP_PORT     = 8100                  # backend/config.py ile aynı

# The work lives here. Checked rather than assumed: writing under /content/drive without a mount
# lands on Colab's own disk instead, and that folder dies with the runtime -- the user believes it
# worked and loses everything later.
DRIVE_ROOT = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs(DRIVE_ROOT, exist_ok=True)   # ilk koşu oluşturur, sonrakiler aynısını kullanır
assert os.path.isdir(DRIVE_ROOT), (
    f"❌ Proje kökü oluşmadı: {DRIVE_ROOT} — Drive bağlanmamış olabilir, hücreyi tekrar çalıştır"
)

# From Colab's Secrets store (🔑 in the left sidebar), never from this cell: the notebook is in
# git, so a token pasted here would sit inside the repository it opens. userdata raises when the
# secret is missing or the notebook was not granted access, and that bare error says nothing --
# so it is caught and the assert below says what to do instead.
try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = ""

assert GITHUB_TOKEN, (
    "❌ GITHUB_TOKEN yok — Colab solundaki 🔑 Secrets panelinden 'GITHUB_TOKEN' adıyla ekle "
    "ve bu deftere erişimi aç (fine-grained, yalnız bu depo, Contents: read)."
)

# The xAI key is deliberately not here. QueenAgent has its own Settings screen, and the key it
# saves lands in settings.json under the root above -- which is on Drive, so it is typed once and
# is still there next session. queen-editor reads it from Secrets; this one does not need to.
print(f"✓ Drive bağlı — proje kökü: {DRIVE_ROOT}")
print(f"✓ Dal: {BRANCH}  |  Depo: {REPO}")
print("✓ GITHUB_TOKEN Secrets'tan okundu")

In [ ]:
# === Clone ===
# Delete-and-reclone: the local tree is disposable and every run starts from the same place. A pull
# can stop on a merge conflict, and a user looking at that has no way to know what it means.
import shutil
import subprocess

# This cell's paths come from CONFIG. Without the gate, a CONFIG that failed stays invisible until
# something much later breaks for a reason nobody can trace back.
assert "CLONE_DIR" in globals(), "❌ Önce CONFIG hücresini çalıştır"


def _mask(text):
    """Replace the token with <token>, so no output ever carries it."""
    return text.replace(GITHUB_TOKEN, "<token>") if GITHUB_TOKEN else text


if os.path.exists(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)

# An argument list, so no shell is involved: a URL that passes through one lands in its history and
# in its log lines. This string carries the token and is never printed.
clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"
result = subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, CLONE_DIR],
    capture_output=True, text=True,
)
if result.returncode != 0:
    # git's own words, masked. A 403 has a dozen causes -- an expired token, a scope that is too
    # narrow, a secret this notebook was never granted, a branch that is gone -- and the notebook
    # knows none of them, so it does not guess.
    raise RuntimeError("❌ Klon başarısız:\n" + _mask(result.stderr.strip() or result.stdout.strip()))

# The app's only third-party dependency. Colab already ships it, so this returns at once -- it is
# written down anyway, because a notebook leaning silently on what Colab happens to include breaks
# with an unreadable ModuleNotFoundError the day that changes.
subprocess.run(["pip", "install", "-q", "flask"], check=True)

# The repo side of this is a test (test_dist_is_committed.py): was the bundle committed. This is the
# other side: did it arrive. Without it a forgotten rebuild reaches the user as a blank page, which
# explains nothing.
DIST = f"{APP_DIR}/frontend/dist/index.html"
assert os.path.exists(DIST), (
    f"❌ Derlenmiş arayüz yok: {DIST} — frontend derlenip commit'lenmemiş (queen-agent/README.md)"
)

print(f"✓ Klon tamam: {CLONE_DIR}  |  dal: {BRANCH}")
print("✓ Flask hazır, derlenmiş arayüz yerinde")